In [ ]:
import pandas as pd

In [ ]:
math_written_df = pd.read_csv('../../data/raw/umimematikucz-system_written_answer_data.csv', delimiter = ';')
math_resource_df = pd.read_csv('../../data/raw/umimematikucz-system_resource_set.csv', delimiter = ';')
inf_binary_df = pd.read_csv('../../data/raw/umimeprogramovatcz-system_binary_choice_data.csv', delimiter = ';')
inf_resource_df = pd.read_csv('../../data/raw/umimeprogramovatcz-system_resource_set.csv', delimiter = ';')

In [ ]:
inf_df = pd.merge(inf_binary_df, inf_resource_df, how='left', left_on='rs', right_on='id')

In [ ]:
math_df = pd.merge(math_written_df, math_resource_df, how='left', left_on='rs', right_on='id')

In [ ]:
math_df.head()

In [ ]:
inf_df.head()

Decode code64 text in qu8estion json into normal text

In [ ]:
from base64 import b64decode
import json

def decode_question(q_str):
    if pd.isna(q_str) or not isinstance(q_str, str):
        return q_str
    try:
        q_list = json.loads(q_str)
        for item in q_list:
            if isinstance(item, list) and len(item) >= 2 and item[0] == 'code64':
                item[1] = b64decode(item[1]).decode('utf-8')
                item[0] = 'code'
        return json.dumps(q_list, ensure_ascii=False)
    except Exception:
        return q_str

def decode_answer(a_str):
    if pd.isna(a_str) or not isinstance(a_str, str):
        return a_str
    try:
        a_list = json.loads(a_str)
        for item in a_list:
            if isinstance(item, list) and len(item) >= 2 and item[0] == 'code64':
                item[1] = b64decode(item[1]).decode('utf-8')
                item[0] = 'code'
        return json.dumps(a_list, ensure_ascii=False)
    except Exception:
        return a_str


# apply to inf_df cols: question, correct, distractor1
inf_df['question_decoded'] = inf_df['question'].apply(decode_question)
inf_df['correct_decoded'] = inf_df['correct'].apply(decode_answer)
inf_df['distractor1_decoded'] = inf_df['distractor1'].apply(decode_answer)


# apply to math_df cols: question
math_df['question_decoded'] = math_df['question'].apply(decode_question)

def extract_text_from_blocks(val, extract_answers=False, only_answers=False):
    if pd.isna(val) or not isinstance(val, str):
        return val
    try:
        blocks = json.loads(val)
        texts = []
        for block in blocks:
            if isinstance(block, list) and len(block) > 1:
                if block[0] == 'input':
                    # Extract answers if requested
                    if extract_answers and isinstance(block[1], dict) and 'answer' in block[1]:
                        ans = block[1]['answer']
                        if isinstance(ans, list):
                            texts.extend([str(a) for a in ans])
                        else:
                            texts.append(str(ans))
                else:
                    # Extract text/images if we are not asking for only answers
                    if not only_answers and isinstance(block[1], (str, int, float)):
                        val_str = str(block[1]).strip()
                        if val_str:
                            texts.append(val_str)
        return "\n".join(texts)
    except Exception:
        return val

# apply text extraction to inf_df
inf_df['question_decoded'] = inf_df['question_decoded'].apply(lambda x: extract_text_from_blocks(x))
inf_df['correct_decoded'] = inf_df['correct_decoded'].apply(lambda x: extract_text_from_blocks(x))
inf_df['distractor1_decoded'] = inf_df['distractor1_decoded'].apply(lambda x: extract_text_from_blocks(x))

# apply text extraction to math_df
# Extract the answer first into correct_decoded
math_df['correct_decoded'] = math_df['question_decoded'].apply(lambda x: extract_text_from_blocks(x, extract_answers=True, only_answers=True))
# overwrite question_decoded with just the text parts
math_df['question_decoded'] = math_df['question_decoded'].apply(lambda x: extract_text_from_blocks(x, extract_answers=False, only_answers=False))

In [ ]:
inf_df.head(10)

In [ ]:
import os
inf_logs_df = pd.DataFrame()
math_logs_df = pd.DataFrame()

for dirpath, dirnames, files in os.walk("../../data/raw/logs/informatics"):
    for file in files:
        inf_temp_df = pd.read_csv(dirpath + '/' + file, encoding = 'utf-8')
        inf_logs_df = pd.concat([inf_logs_df,inf_temp_df], axis = 0)

for dirpath, dirnames, files in os.walk("../../data/raw/logs/math"):
    for file in files:
        math_temp_df = pd.read_csv(dirpath + '/' + file, encoding = 'utf-8')
        math_logs_df = pd.concat([math_logs_df,math_temp_df], axis = 0)


In [ ]:
inf_logs_df.head(10)

In [ ]:
def build_item_metrics(logs_df, item_col='item'):
    if logs_df.empty or item_col not in logs_df.columns:
        return pd.DataFrame(columns=[item_col, 'avg_response_time', 'correct_rate', 'number_of_occurrences'])

    response_time = pd.to_numeric(logs_df.get('responseTime'), errors='coerce')
    correct_numeric = pd.to_numeric(logs_df.get('correct'), errors='coerce')

    metrics_df = (
        logs_df.assign(responseTime_num=response_time, correct_num=correct_numeric)
        .groupby(item_col, dropna=False)
        .agg(
            avg_response_time=('responseTime_num', 'mean'),
            correct_rate=('correct_num', 'mean'),
            number_of_occurrences=('correct_num', 'size'),
        )
        .reset_index()
    )
    return metrics_df

inf_item_metrics = build_item_metrics(inf_logs_df)
math_item_metrics = build_item_metrics(math_logs_df)

inf_df = inf_df.merge(inf_item_metrics, how='left', left_on='id_x', right_on='item')
math_df = math_df.merge(math_item_metrics, how='left', left_on='id_x', right_on='item')


In [ ]:
inf_item_metrics

In [ ]:
inf_df

In [ ]:
math_df

In [ ]:
inf_df.to_csv('../../data/processed/informatics.csv', encoding='utf-8', sep = ';')
math_df.to_csv('../../data/processed/math.csv', encoding='utf-8', sep = ';')